In [1]:
# !pip install duckdb

In [2]:
import duckdb
from pathlib import Path
import pandas as pd

In [3]:
PROJECT_ROOT = Path.cwd()

# Data directories
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
WAREHOUSE_DIR = DATA_DIR / "warehouse"

WAREHOUSE_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH = RAW_DIR / "CompaniesHouseData-2026-03-02.csv"
CSV_PATH = PROCESSED_DIR / "entity_master_v1.csv"
DUCKDB_PATH = WAREHOUSE_DIR / "project.duckdb"

print("CSV exists:", CSV_PATH.exists())
print("DuckDB path:", DUCKDB_PATH)

CSV exists: True
DuckDB path: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/data/warehouse/project.duckdb


In [4]:
duckdb_con = duckdb.connect(str(DUCKDB_PATH))
print("Connected to:", DUCKDB_PATH)

Connected to: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/data/warehouse/project.duckdb


In [5]:
duckdb_con.execute(f"""
    CREATE OR REPLACE TABLE entity_master_v1 AS
    SELECT *
    FROM read_csv_auto('{CSV_PATH}');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
duckdb_con.execute("""
    SELECT *
    FROM entity_master_v1
    LIMIT 10
""").fetchdf()

,entity_id,entity_name,company_category,company_status,country_of_origin,incorporation_date,account_category,sic_text_1,sic_text_2,sic_text_3,sic_text_4,post_town,county,country,postcode,source_uri,entity_name_raw
0,16873705,FE NETWORK LIMITED,Private Limited Company,Active,United Kingdom,2025-11-25,NO ACCOUNTS FILED,86210 - General medical practice activities,None,None,None,"HARLOW, ESSEX",None,UNITED KINGDOM,CM20 1YS,http://business.data.gov.uk/id/company/16873705,!FE NETWORK LIMITED
1,15073164,NFLECTION ADVISORY LIMITED,Private Limited Company,Active,United Kingdom,2023-08-15,TOTAL EXEMPTION FULL,70229 - Management consultancy activities othe...,None,None,None,POTTERS BAR,HERTFORDSHIRE,ENGLAND,EN6 2DA,http://business.data.gov.uk/id/company/15073164,!NFLECTION ADVISORY LIMITED
2,13522064,NFOGENIE LTD,Private Limited Company,Active,United Kingdom,2021-07-21,MICRO ENTITY,58290 - Other software publishing,None,None,None,LONDON,GREATER LONDON,UNITED KINGDOM,WC2H 9JQ,http://business.data.gov.uk/id/company/13522064,!NFOGENIE LTD
3,11006939,NNOV8 LIMITED,Private Limited Company,Active,United Kingdom,2017-10-11,MICRO ENTITY,62090 - Other information technology service a...,70229 - Management consultancy activities othe...,None,None,EDENBRIDGE,None,ENGLAND,TN8 5NF,http://business.data.gov.uk/id/company/11006939,!NNOV8 LIMITED
4,SC606050,NSPIRED INVESTMENTS LTD,Private Limited Company,Active,United Kingdom,2018-08-22,TOTAL EXEMPTION FULL,68209 - Other letting and operating of own or ...,None,None,None,ABERDEEN,None,SCOTLAND,AB11 7SY,http://business.data.gov.uk/id/company/SC606050,!NSPIRED INVESTMENTS LTD
5,07687209,OBAC UK LIMITED,Private Limited Company,Active,United Kingdom,2011-06-29,TOTAL EXEMPTION FULL,70229 - Management consultancy activities othe...,None,None,None,TADLEY,HAMPSHIRE,None,RG26 5AT,http://business.data.gov.uk/id/company/07687209,!OBAC UK LIMITED
6,13310195,""" TRIPLE D"" PROPERTIES LIMITED",Private Limited Company,Active,United Kingdom,2021-04-01,UNAUDITED ABRIDGED,68209 - Other letting and operating of own or ...,68320 - Management of real estate on a fee or ...,None,None,BIRMINGHAM,None,ENGLAND,B24 9NB,http://business.data.gov.uk/id/company/13310195,""" TRIPLE D"" PROPERTIES LIMITED"
7,11303802,"""1ST RATE"" PSYCHOLOGY SERVICES LTD",Private Limited Company,Active,United Kingdom,2018-04-11,MICRO ENTITY,85600 - Educational support services,86900 - Other human health activities,None,None,GREAT WAKERING,ESSEX,UNITED KINGDOM,SS3 0GW,http://business.data.gov.uk/id/company/11303802,"""1ST RATE"" PSYCHOLOGY SERVICES LTD"
8,10694769,"""786"" MAZ OFFICE SUPPORT LIMITED",Private Limited Company,Active,United Kingdom,2017-03-28,MICRO ENTITY,82110 - Combined office administrative service...,None,None,None,PRESTON,None,ENGLAND,PR2 9QL,http://business.data.gov.uk/id/company/10694769,"""786"" MAZ OFFICE SUPPORT LIMITED"
9,15761044,"""A TASTE OF TUSCANY"" LTD",Private Limited Company,Active,United Kingdom,2024-06-04,NO ACCOUNTS FILED,56101 - Licensed restaurants,68209 - Other letting and operating of own or ...,None,None,LONDON,None,UNITED KINGDOM,WC2H 9JQ,http://business.data.gov.uk/id/company/15761044,"""A TASTE OF TUSCANY"" LTD"


In [7]:
duckdb_con.execute(f"""
CREATE OR REPLACE TABLE entity_mortgage_features AS
SELECT
    "CompanyNumber" AS entity_id,
    "Mortgages.NumMortCharges" AS num_mort_charges,
    "Mortgages.NumMortOutstanding" AS num_mort_outstanding,
    "Mortgages.NumMortPartSatisfied" AS num_mort_part_satisfied,
    "Mortgages.NumMortSatisfied" AS num_mort_satisfied
FROM read_csv_auto('{RAW_DATA_PATH}')
""")

In [8]:
duckdb_con.execute("""
    SELECT *
    FROM entity_mortgage_features
    LIMIT 10
""").fetchdf()

,entity_id,num_mort_charges,num_mort_outstanding,num_mort_part_satisfied,num_mort_satisfied
0,08209948,0,0,0,0
1,11743365,0,0,0,0
2,16873705,0,0,0,0
3,15073164,0,0,0,0
4,13522064,0,0,0,0
5,11006939,0,0,0,0
6,SC606050,5,5,0,0
7,SC421617,0,0,0,0
8,FC031362,0,0,0,0
9,07687209,1,0,0,1


In [9]:
duckdb_con.execute("""
CREATE OR REPLACE TABLE entity_signal_base AS
SELECT
    master.*,
    mortgage.num_mort_charges,
    mortgage.num_mort_outstanding,
    mortgage.num_mort_part_satisfied,
    mortgage.num_mort_satisfied
FROM entity_master_v1 master
LEFT JOIN entity_mortgage_features mortgage
    ON master.entity_id = mortgage.entity_id
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
duckdb_con.execute("""
    SELECT *
    FROM entity_signal_base
    LIMIT 10
""").fetchdf()

,entity_id,entity_name,company_category,company_status,country_of_origin,incorporation_date,account_category,sic_text_1,sic_text_2,sic_text_3,...,post_town,county,country,postcode,source_uri,entity_name_raw,num_mort_charges,num_mort_outstanding,num_mort_part_satisfied,num_mort_satisfied
0,13293393,BOYCE SPV 2 LTD,Private Limited Company,Active,United Kingdom,2021-03-26,UNAUDITED ABRIDGED,68100 - Buying and selling of own real estate,68209 - Other letting and operating of own or ...,None,...,NEWPORT,None,WALES,NP20 1EB,http://business.data.gov.uk/id/company/13293393,BOYCE SPV 2 LTD,1,1,0,0
1,13356770,BOYCE SPV 3 LTD,Private Limited Company,Active,United Kingdom,2021-04-26,MICRO ENTITY,68100 - Buying and selling of own real estate,68209 - Other letting and operating of own or ...,None,...,NEWPORT,None,WALES,NP20 1EB,http://business.data.gov.uk/id/company/13356770,BOYCE SPV 3 LTD,1,1,0,0
2,14011296,BOYCE SPV 4 LTD,Private Limited Company,Active,United Kingdom,2022-03-30,MICRO ENTITY,68100 - Buying and selling of own real estate,68209 - Other letting and operating of own or ...,None,...,NEWPORT,None,WALES,NP20 1EB,http://business.data.gov.uk/id/company/14011296,BOYCE SPV 4 LTD,2,1,0,1
3,14011310,BOYCE SPV 5 LTD,Private Limited Company,Active,United Kingdom,2022-03-30,MICRO ENTITY,68100 - Buying and selling of own real estate,68209 - Other letting and operating of own or ...,None,...,NEWPORT,None,WALES,NP20 1EB,http://business.data.gov.uk/id/company/14011310,BOYCE SPV 5 LTD,1,1,0,0
4,14009425,BOYCE SPV 6 LTD,Private Limited Company,Active,United Kingdom,2022-03-29,MICRO ENTITY,68100 - Buying and selling of own real estate,68209 - Other letting and operating of own or ...,None,...,NEWPORT,None,WALES,NP20 1EB,http://business.data.gov.uk/id/company/14009425,BOYCE SPV 6 LTD,1,0,0,1
5,14240411,BOYCE SPV 7 LTD,Private Limited Company,Active,United Kingdom,2022-07-18,MICRO ENTITY,68100 - Buying and selling of own real estate,68209 - Other letting and operating of own or ...,None,...,NEWPORT,None,WALES,NP20 1EB,http://business.data.gov.uk/id/company/14240411,BOYCE SPV 7 LTD,1,1,0,0
6,13289867,BOYCE SPV LTD,Private Limited Company,Active,United Kingdom,2021-03-24,MICRO ENTITY,68209 - Other letting and operating of own or ...,None,None,...,NEWPORT,None,WALES,NP20 1EB,http://business.data.gov.uk/id/company/13289867,BOYCE SPV LTD,1,1,0,0
7,03817365,BOYCE THORNTON LIMITED,Private Limited Company,Active,United Kingdom,1999-07-30,TOTAL EXEMPTION FULL,68310 - Real estate agencies,None,None,...,SURREY,None,None,KT22 0JP,http://business.data.gov.uk/id/company/03817365,BOYCE THORNTON LIMITED,1,0,0,1
8,05333757,BOYCE TRENCHARD LIMITED,Private Limited Company,Active,United Kingdom,2005-01-17,MICRO ENTITY,69201 - Accounting and auditing activities,None,None,...,EASTBOURNE,EAST SUSSEX,ENGLAND,BN20 9JS,http://business.data.gov.uk/id/company/05333757,BOYCE TRENCHARD LIMITED,0,0,0,0
9,12481610,BOYCE UTILITIES LIMITED,Private Limited Company,Active,United Kingdom,2020-02-25,TOTAL EXEMPTION FULL,42990 - Construction of other civil engineerin...,None,None,...,PRESTON,LANCASHIRE,UNITED KINGDOM,PR2 9WT,http://business.data.gov.uk/id/company/12481610,BOYCE UTILITIES LIMITED,0,0,0,0


In [11]:
duckdb_con.execute("""
CREATE OR REPLACE TABLE entity_risk_signals_v1 AS
SELECT
    entity_id,
    entity_name,
    company_status,
    company_category,
    incorporation_date,
    account_category,
    sic_text_1,
    post_town,
    country,
    postcode,
    num_mort_charges,
    num_mort_outstanding,
    num_mort_part_satisfied,
    num_mort_satisfied,

    CASE
        WHEN incorporation_date >= current_date - INTERVAL 365 DAY
        THEN 1 ELSE 0
    END AS new_entity_flag,


    CASE
        WHEN post_town IS NULL OR postcode IS NULL
        THEN 1 ELSE 0
    END AS missing_location_flag,


    CASE
        WHEN account_category = 'NO ACCOUNTS FILED'
        THEN 1 ELSE 0
    END AS no_accounts_filled_flag,                 


    CASE
        WHEN coalesce(num_mort_outstanding, 0) > 0
        THEN 1 ELSE 0
    END AS has_outstanding_mortgage_flag,

                   
    CASE
        WHEN coalesce(num_mort_outstanding, 0) > 0
        AND (
            coalesce(num_mort_part_satisfied, 0) > 0
            OR coalesce(num_mort_satisfied, 0) > 0
        )
        THEN 1 ELSE 0
    END AS mixed_mortgage_profile_flag,

FROM entity_signal_base
""")

In [12]:
duckdb_con.execute("""
    SELECT * 
    FROM entity_risk_signals_v1
    LIMIT 10
""").fetchdf()

,entity_id,entity_name,company_status,company_category,incorporation_date,account_category,sic_text_1,post_town,country,postcode,num_mort_charges,num_mort_outstanding,num_mort_part_satisfied,num_mort_satisfied,new_entity_flag,missing_location_flag,no_accounts_filled_flag,has_outstanding_mortgage_flag,mixed_mortgage_profile_flag
0,13293393,BOYCE SPV 2 LTD,Active,Private Limited Company,2021-03-26,UNAUDITED ABRIDGED,68100 - Buying and selling of own real estate,NEWPORT,WALES,NP20 1EB,1,1,0,0,0,0,0,1,0
1,13356770,BOYCE SPV 3 LTD,Active,Private Limited Company,2021-04-26,MICRO ENTITY,68100 - Buying and selling of own real estate,NEWPORT,WALES,NP20 1EB,1,1,0,0,0,0,0,1,0
2,14011296,BOYCE SPV 4 LTD,Active,Private Limited Company,2022-03-30,MICRO ENTITY,68100 - Buying and selling of own real estate,NEWPORT,WALES,NP20 1EB,2,1,0,1,0,0,0,1,1
3,14011310,BOYCE SPV 5 LTD,Active,Private Limited Company,2022-03-30,MICRO ENTITY,68100 - Buying and selling of own real estate,NEWPORT,WALES,NP20 1EB,1,1,0,0,0,0,0,1,0
4,14009425,BOYCE SPV 6 LTD,Active,Private Limited Company,2022-03-29,MICRO ENTITY,68100 - Buying and selling of own real estate,NEWPORT,WALES,NP20 1EB,1,0,0,1,0,0,0,0,0
5,14240411,BOYCE SPV 7 LTD,Active,Private Limited Company,2022-07-18,MICRO ENTITY,68100 - Buying and selling of own real estate,NEWPORT,WALES,NP20 1EB,1,1,0,0,0,0,0,1,0
6,13289867,BOYCE SPV LTD,Active,Private Limited Company,2021-03-24,MICRO ENTITY,68209 - Other letting and operating of own or ...,NEWPORT,WALES,NP20 1EB,1,1,0,0,0,0,0,1,0
7,03817365,BOYCE THORNTON LIMITED,Active,Private Limited Company,1999-07-30,TOTAL EXEMPTION FULL,68310 - Real estate agencies,SURREY,None,KT22 0JP,1,0,0,1,0,0,0,0,0
8,05333757,BOYCE TRENCHARD LIMITED,Active,Private Limited Company,2005-01-17,MICRO ENTITY,69201 - Accounting and auditing activities,EASTBOURNE,ENGLAND,BN20 9JS,0,0,0,0,0,0,0,0,0
9,12481610,BOYCE UTILITIES LIMITED,Active,Private Limited Company,2020-02-25,TOTAL EXEMPTION FULL,42990 - Construction of other civil engineerin...,PRESTON,UNITED KINGDOM,PR2 9WT,0,0,0,0,0,0,0,0,0


In [13]:
duckdb_con.execute("""
CREATE OR REPLACE TABLE entity_risk_signals_v2 AS
SELECT
    *,
    new_entity_flag
    + missing_location_flag
    + no_accounts_filled_flag
    + has_outstanding_mortgage_flag
    + mixed_mortgage_profile_flag
    AS review_priority_score,

    CASE
        WHEN (
            new_entity_flag
            + missing_location_flag
            + no_accounts_filled_flag
            + has_outstanding_mortgage_flag
            + mixed_mortgage_profile_flag
        ) >= 4 THEN 'High'
        
        WHEN (
            new_entity_flag
            + missing_location_flag
            + no_accounts_filled_flag
            + has_outstanding_mortgage_flag
            + mixed_mortgage_profile_flag
        ) = 3 THEN 'Medium'
        
        ELSE 'Low'
    END AS review_priority_band

FROM entity_risk_signals_v1
""")

In [14]:
duckdb_con.execute("""
    SELECT review_priority_band, COUNT(*) AS n
    FROM entity_risk_signals_v2
    GROUP BY review_priority_band
    ORDER BY n DESC
""").fetchdf()

,review_priority_band,n
0,Low,4172483
1,Medium,18837
2,High,215


In [15]:
duckdb_con.execute("""
    SELECT
        entity_id,
        entity_name,
        account_category,
        sic_text_1,
        incorporation_date,
        new_entity_flag,
        missing_location_flag,
        no_accounts_filled_flag,
        num_mort_charges,
        num_mort_outstanding,
        has_outstanding_mortgage_flag,
        mixed_mortgage_profile_flag,
        review_priority_score,
        review_priority_band
    FROM entity_risk_signals_v2
    WHERE review_priority_band = 'High'
    ORDER BY review_priority_score DESC, entity_name
    LIMIT 20
""").fetchdf()

,entity_id,entity_name,account_category,sic_text_1,incorporation_date,new_entity_flag,missing_location_flag,no_accounts_filled_flag,num_mort_charges,num_mort_outstanding,has_outstanding_mortgage_flag,mixed_mortgage_profile_flag,review_priority_score,review_priority_band
0,16502889,... AND RELAX COTTAGES LTD,NO ACCOUNTS FILED,68209 - Other letting and operating of own or ...,2025-06-07,1,0,1,2,1,1,1,4,High
1,00156048,00156048 LIMITED,NO ACCOUNTS FILED,None Supplied,1919-06-13,0,1,1,2,1,1,1,4,High
2,01115163,01115163 LIMITED,NO ACCOUNTS FILED,None Supplied,1973-05-24,0,1,1,9,6,1,1,4,High
3,16435051,360SQUARE LTD,NO ACCOUNTS FILED,68100 - Buying and selling of own real estate,2025-05-07,1,0,1,3,1,1,1,4,High
4,16330496,50 HIGH STREET SOUTH RUSHDEN LTD,NO ACCOUNTS FILED,68100 - Buying and selling of own real estate,2025-03-20,1,0,1,3,1,1,1,4,High
5,16531177,50-52 CR SUBCO LIMITED,NO ACCOUNTS FILED,41100 - Development of building projects,2025-06-20,1,0,1,3,2,1,1,4,High
6,16392049,AD3 PROPERTIES LTD,NO ACCOUNTS FILED,68209 - Other letting and operating of own or ...,2025-04-16,1,0,1,3,2,1,1,4,High
7,16325942,AKKTION PROPERTIES LIMITED,NO ACCOUNTS FILED,68209 - Other letting and operating of own or ...,2025-03-19,1,0,1,2,1,1,1,4,High
8,SC029423,ALEXANDER SUTHERLAND LIMITED,NO ACCOUNTS FILED,None Supplied,1953-05-01,0,1,1,2,1,1,1,4,High
9,16484242,ALOEHAWK LIMITED,NO ACCOUNTS FILED,68209 - Other letting and operating of own or ...,2025-05-30,1,0,1,6,3,1,1,4,High


In [16]:
duckdb_con.execute("""
    COPY entity_risk_signals_v2
    TO 'data/processed/entity_risk_signals_v2.csv'
    (HEADER, DELIMITER ',');
""")